#  Recommendation de film basée sur le contenu  (Content-Based)


In [ ]:
import numpy as np
import pandas as pd
import sklearn
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [ ]:
ratings = pd.read_csv("ratings.csv")
movies = pd.read_csv("movies.csv")

## Le but est de construre  un moteur de recommandation basé sur le contenu qui calcule la similarité entre les films en fonction de leur genre. Il suggérera les films les plus similaires à un film particulier en fonction de son genre. Pour ce faire, on utilise le fichier movies.csv.

In [ ]:
# Décomposer la chaîne de caractère "genre" en un tableau de chaînes de caractères
movies['genres'] = movies['genres'].str.split('|')
# Convertire genres en string value
movies['genres'] = movies['genres'].fillna("").astype('str')

## La fonction TfidfVectorizer de scikit-learn transforme le texte en vecteurs de caractéristiques pouvant être utilisés comme entrée de l'estimateur

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
tf = TfidfVectorizer(analyzer='word',ngram_range=(1, 2),min_df=1, stop_words='english')
tfidf_matrix = tf.fit_transform(movies['genres'])
tfidf_matrix.shape

## On utilise ensuite **[Cosine Similarity] pour calculer pour calculer une quantité numérique qui indique la similarité entre deux films. Puisque nous avons utilisé le vecteur TF-IDF, le calcul du produit de points nous donnera directement le score de similarité cosinus. Par conséquent, nous utiliserons le **noyau_linéaire** de sklearn au lieu de cosinus_similarities car il est beaucoup plus rapide.

In [ ]:
from sklearn.metrics.pairwise import linear_kernel
cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)
cosine_sim[:4, :4]

## Remarque: Si vous rencontrez des problèmes de mémoire ou d'exécution prolongée, vous pouvez calculer uniquement les similarités pour un sous-ensemble d'éléments ou pour les k-plus proches voisins

In [ ]:
from sklearn.metrics.pairwise import linear_kernel
import numpy as np

# Exemple : calculer uniquement les similarités pour un échantillon
subset_indices = np.random.choice(tfidf_matrix.shape[0], size=100, replace=False)
subset_tfidf = tfidf_matrix[subset_indices]
cosine_sim_subset = linear_kernel(subset_tfidf, tfidf_matrix)


## On dispose à présent d'une matrice de similarité cosinus par paire pour tous les films du jeu de données. L'étape suivante consiste à écrire une fonction qui renvoie les 20 films les plus similaires sur la base du score de similarité cosinus.

In [ ]:

titles = movies['title']
indices = pd.Series(movies.index, index=movies['title'])


def genre_recommendations(title):
    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:21]
    movie_indices = [i[0] for i in sim_scores]
    return titles.iloc[movie_indices]

## Essayons d'obtenir les meilleures recommandations pour quelques films et voyons si elles sont bonnes.

In [ ]:
genre_recommendations('Good Will Hunting (1997)').head(20)

In [ ]:
genre_recommendations('Toy Story (1995)').head(20)

In [ ]:
genre_recommendations('Saving Private Ryan (1998)').head(20)

On obtient une liste de recommandations pas trop mauvaise pour **Good Will Hunting** (Drame), **Toy Story** (Animation, Enfants, Comédie), et **Saving Private Ryan** (Action, Thriller, Guerre).

Dans l'ensemble, voici les avantages de la recommandation basée sur le contenu :
* Pas besoin de données sur les autres utilisateurs, donc pas de problèmes de démarrage à froid ou de rareté.
* Peut recommander aux utilisateurs ayant des goûts uniques.
* Possibilité de recommander des articles nouveaux et impopulaires.
* Possibilité de fournir des explications sur les éléments recommandés en énumérant les caractéristiques du contenu à l'origine de la recommandation (dans le cas présent, les genres de films).

Cette approche présente toutefois certains inconvénients :
* Il est difficile de trouver les caractéristiques appropriées.
* Elle ne permet pas de recommander des éléments en dehors du profil de contenu de l'utilisateur.
* Impossibilité d'exploiter les jugements de qualité des autres utilisateurs.


# Exercice  « Recommandé pour vous »

La fonction `genre_recommendations(title)` répond à la question « quels films ressemblent à *Toy Story* ? ». La page d'accueil d'une plateforme de streaming doit répondre à une autre question : **« quels films proposer à *cet* utilisateur ? »**

Votre mission :
1. construire le **profil de goûts** d'un utilisateur à partir des films qu'il a aimés ;
2. lui **recommander 10 films** qu'il n'a pas encore vus ;
3. **vérifier** si votre moteur fait mieux que la stratégie la plus simple : montrer à tout le monde les films les plus populaires.


On réutilise `movies`, `ratings`, `tf` et `tfidf_matrix`, construits plus haut.
Remplacez chaque `...` par votre code.

### Partie 1 : le profil d'un utilisateur (15 min)

Un utilisateur qui a aimé *Toy Story* et *Shrek* a des goûts « Animation, Children, Comedy ». On représente ces goûts par **la moyenne des vecteurs TF-IDF des films qu'il a aimés** (note ≥ 4). Ce vecteur moyen est son **profil**.

Rappel : la ligne `i` de `tfidf_matrix` correspond à la ligne `i` de `movies` (même ordre).

**1.1** Récupérez les `movieId` des films que l'utilisateur 1 a notés 4 ou plus.

In [ ]:
user_id = 1

# TODO : les movieId des films notés 4 ou plus par l'utilisateur user_id
films_aimes = ...

print(len(films_aimes), "films aimés")   # attendu : 200
movies[movies['movieId'].isin(films_aimes)].head()

**1.2** Construisez le profil : la moyenne des lignes de `tfidf_matrix` qui correspondent à ces films.

*Aide* : une matrice creuse se découpe comme un tableau NumPy (`tfidf_matrix[lignes]`) et a une méthode `.mean(axis=0)` (la moyenne de chaque colonne).

In [ ]:
# Les numéros de ligne de ces films dans movies, donc dans tfidf_matrix
lignes = np.flatnonzero(movies['movieId'].isin(films_aimes))

# TODO : la moyenne des lignes `lignes` de tfidf_matrix
profil = ...

profil = np.asarray(profil).reshape(1, -1)   # un vecteur ligne : 1 × nombre de termes
profil.shape   # attendu : (1, 177)

**1.3** Expliquez le profil : affichez les 5 termes qui pèsent le plus. Un terme est un genre seul (`comedy`) ou une paire de genres (`action adventure`), car le `TfidfVectorizer` a été créé avec `ngram_range=(1, 2)`. Le profil correspond-il aux films aimés ?

In [ ]:
termes = tf.get_feature_names_out()              # le terme de chaque colonne de tfidf_matrix
poids = pd.Series(profil.ravel(), index=termes)

# TODO : les 5 termes de plus fort poids
...

### Partie 2 : recommander 10 films (15 min)

Le score d'un film mesure sa ressemblance avec le profil. Comme pour `cosine_sim` plus haut, `linear_kernel(profil, tfidf_matrix)` calcule d'un coup un score pour chaque film. Il reste à retirer les films que l'utilisateur a déjà notés et à garder les 10 meilleurs.

Complétez la fonction `recommander_pour`. Elle reçoit les notes en paramètre, car la partie 3 l'appellera avec une partie seulement des notes.

*Aide* :
* `linear_kernel` renvoie un tableau 2D (1 × nombre de films) ; `.ravel()` le met à plat ;
* `np.argsort(-scores, kind='stable')` renvoie les positions des films, du plus grand score au plus petit. À score égal, l'ordre du fichier est conservé, donc tout le monde obtient le même résultat.

In [ ]:
def recommander_pour(user_id, notes, n=10):
    """Les movieId des n films les plus proches du profil de user_id, hors films déjà notés."""
    notes_user = notes[notes['userId'] == user_id]
    films_aimes = notes_user.loc[notes_user['rating'] >= 4, 'movieId']
    lignes = np.flatnonzero(movies['movieId'].isin(films_aimes))
    if len(lignes) == 0:
        return []   # aucun film aimé, donc pas de profil (voir la partie 4)
    profil = np.asarray(tfidf_matrix[lignes].mean(axis=0)).reshape(1, -1)
    deja_vus = movies['movieId'].isin(notes_user['movieId']).to_numpy()   # True pour les films déjà notés

    # TODO 1 : un score par film (linear_kernel entre profil et tfidf_matrix), mis à plat
    scores = ...

    # TODO 2 : un score de -1 pour les films déjà notés (deja_vus), pour ne pas les recommander
    ...

    # TODO 3 : les positions des n meilleurs scores, puis la liste de leurs movieId
    #          (movies['movieId'].iloc[positions].tolist())
    ...


def afficher(movie_ids):
    """Le titre et les genres d'une liste de movieId, dans l'ordre de la liste."""
    return movies.set_index('movieId').loc[list(movie_ids), ['title', 'genres']]


afficher(recommander_pour(1, ratings))

**2.1** Essayez avec d'autres utilisateurs, par exemple 401, 533 et 267. Les recommandations correspondent-elles à leurs goûts ?

In [ ]:
afficher(recommander_pour(401, ratings))   # changez l'utilisateur : 533, 267…

**2.2** La cellule suivante affiche les 30 meilleurs scores de l'utilisateur 1. Beaucoup sont exactement égaux : pourquoi ? Qu'est-ce que cela change pour le choix des 10 films ?

In [ ]:
# Les 30 meilleurs scores de l'utilisateur 1 (son profil vient de la partie 1)
scores_1 = linear_kernel(profil, tfidf_matrix).ravel()
np.sort(scores_1)[::-1][:30].round(3)

### Partie 3 : faites-vous mieux que « les plus populaires » ? (20 min)

Les recommandations ont l'air cohérentes, mais sont-elles **utiles** ? Pour le savoir, on simule l'avenir :
1. on cache 20 % des notes (le **test**) ;
2. on recommande 10 films à chaque utilisateur à partir des 80 % restants (le **train**) ;
3. on compte combien de ces 10 films l'utilisateur a **aimés** (note ≥ 4) dans ses notes cachées.

C'est la **precision@10** : sur les 10 films affichés sur sa page d'accueil, quelle part l'utilisateur a-t-il vraiment aimée ? 3 films aimés sur 10 donnent 0,3.

On compare votre moteur à une **baseline**, la stratégie la plus simple possible : recommander à tout le monde les films les plus notés, sauf ceux qu'il a déjà vus. Un modèle qui ne bat pas sa baseline ne mérite pas d'être mis en production.

In [ ]:
from sklearn.model_selection import train_test_split

notes_train, notes_test = train_test_split(ratings, test_size=0.2, random_state=42)

# Pour chaque utilisateur, l'ensemble des films qu'il a aimés (note >= 4) parmi ses notes cachées
aimes_test = notes_test[notes_test['rating'] >= 4].groupby('userId')['movieId'].apply(set)
print(len(aimes_test), "utilisateurs ont au moins un film aimé dans le test")

**3.1** Complétez `precision_at_k` et `recommander_populaires`, puis vérifiez le résultat.

In [ ]:
def precision_at_k(recos, aimes, k=10):
    """La part des k premières recommandations que l'utilisateur a aimées (aimes est un set)."""

    # TODO : le nombre de films de recos[:k] qui sont dans aimes, divisé par k
    ...



# Les movieId du train, du plus noté au moins noté
popularite = notes_train['movieId'].value_counts()

def recommander_populaires(user_id, notes, n=10):
    """Les n films les plus notés que user_id n'a pas encore notés."""
    deja_vus = set(notes.loc[notes['userId'] == user_id, 'movieId'])

    # TODO : parcourir popularite.index et garder les n premiers films qui ne sont pas dans deja_vus
    ...



print(precision_at_k([1, 2, 3, 4], {2, 4, 99}, k=4))   # attendu : 0.5
afficher(recommander_populaires(1, notes_train))

**3.2** La cellule suivante calcule, pour chaque stratégie, la precision@10 moyenne sur tous les utilisateurs, et le nombre de films différents recommandés au total (la **couverture** du catalogue). Quelle stratégie gagne ?

In [ ]:
def evaluer(strategies):
    """La precision@10 moyenne et la couverture du catalogue de chaque stratégie."""
    bilan = []
    for nom, recommander in strategies.items():
        recos = {u: recommander(u, notes_train) for u in aimes_test.index}
        bilan.append({
            'stratégie': nom,
            'precision@10': np.mean([precision_at_k(recos[u], aimes_test[u]) for u in aimes_test.index]),
            'films différents recommandés': len({m for liste in recos.values() for m in liste}),
        })
    return pd.DataFrame(bilan).set_index('stratégie').round(3)


evaluer({'contenu': recommander_pour, 'popularité': recommander_populaires})

### Partie 4 : questions (10 min)

1. **Pourquoi la popularité gagne-t-elle ?** *Pistes* : combien de genres différents existe-t-il ? Que sait le moteur de contenu de la qualité d'un film ? Et la stratégie gagnante est-elle bonne pour la plateforme, si elle montre les mêmes films à tout le monde ?
2. **Démarrage à froid.** Plus haut, on lit que le filtrage basé sur le contenu n'a « pas de problèmes de démarrage à froid ». Vérifiez-le dans deux cas : (a) un nouvel utilisateur qui n'a encore rien noté ; (b) un nouveau film que personne n'a encore noté. Que peut proposer chaque stratégie ?
3. **Onboarding.** À l'inscription, le nouvel utilisateur coche les genres qu'il aime. Construisez son profil avec `tf.transform([...])` et recommandez-lui 10 films (cellule ci-dessous).
4. **Bonus : un moteur hybride.** Multipliez le score de contenu par `np.log1p(nombre de notes du film)` : à genres égaux, les films les plus vus passent devant. La precision@10 s'améliore-t-elle ? Et la couverture ?

In [ ]:
genres_coches = "Animation Children Comedy"      # les genres cochés à l'inscription
profil_nouveau = tf.transform([genres_coches])

# TODO : un score par film pour ce profil, puis les movieId des 10 meilleurs (comme dans recommander_pour)
...

In [ ]:
# Bonus : le nombre de notes de chaque film dans le train, dans l'ordre des lignes de movies
nb_notes = movies['movieId'].map(notes_train['movieId'].value_counts()).fillna(0).to_numpy()

# TODO : recopiez recommander_pour sous le nom recommander_hybride,
#        en multipliant les scores par np.log1p(nb_notes)
...

# TODO : comparez les trois stratégies
# evaluer({'contenu': recommander_pour, 'popularité': recommander_populaires, 'hybride': recommander_hybride})

### À retenir

* Le **profil** d'un utilisateur est la moyenne des vecteurs des films qu'il a aimés : on passe ainsi de « films similaires à X » à « recommandé pour vous ».
* On évalue un moteur de recommandation sur ce que l'utilisateur voit, le top 10 : c'est la **precision@10**, mesurée en cachant une partie des notes.
* Comparez toujours votre modèle à une **baseline** simple, ici « les plus populaires ». Elle est souvent difficile à battre.
* La precision ne suffit pas : regardez aussi la **couverture**, le nombre de films différents recommandés.
* Pour le mini-projet, réutilisez ce découpage train/test, `precision_at_k` et la baseline pour comparer vos approches.

## Collaborative Filtering Recommendation Model
Dans ce qui suit et en complément du TP1, nous allons utiliser ici le filtrage collaboratif basé sur la mémoire pour faire des recommandations aux utilisateurs de films.
L'idée est de construire une matrice de similarité.  la **matrice de similarité utilisateur** se compose de certaines mesures de distance qui mesurent la similarité entre deux paires d'utilisateurs. De même, la **matrice de similarité des items** mesure la similarité entre deux paires d'items.

Trois mesures de similarité de distance sont généralement utilisées dans le cadre du filtrage collaboratif :
1. **Similarité Jaccard** :
    * La similarité est basée sur le nombre d'utilisateurs qui ont évalué les éléments A et B divisé par le nombre d'utilisateurs qui ont évalué soit A soit B
    * Elle est typiquement utilisée lorsque nous n'avons pas d'évaluation numérique mais juste une valeur booléenne, comme l'achat d'un produit ou le clic sur une annonce.

2. **la similarité cosinus** : (comme dans le système basé sur le contenu)
    * La similarité est le cosinus de l'angle entre les 2 vecteurs des vecteurs des articles A et B.
    * Plus les vecteurs sont proches, plus l'angle est petit et plus le cosinus est grand.

3. **Similarité de Pearson** :
    * La similarité est le coefficient de Pearson entre les deux vecteurs.

Nous allons tester la **similarité de Pearson** dans cette mise en œuvre.

### Mise en œuvre
On utilise d'abord le fichier **ratings.csv** car il contient l'identifiant de l'utilisateur, l'identifiant du film et les évaluations. Ces trois éléments sont tout ce dont on a besoin pour déterminer la similarité des utilisateurs sur la base de leurs évaluations d'un film particulier.

On commence par un  traitement rapide des données :

In [ ]:
# remplir les valeurs NaN avec des 0
ratings['userId'] = ratings['userId'].fillna(0)
ratings['movieId'] = ratings['movieId'].fillna(0)

# Remplacer les valeurs NaN dans la colonne de notation par la moyenne de toutes les valeurs
ratings['rating'] = ratings['rating'].fillna(ratings['rating'].mean())

En raison de la puissance de calcul limitée, nous n'utiliiserons qu'un échantillon aléatoire de 20 000 évaluations (2 %) sur les 1 million d'évaluations.

In [ ]:

small_data = ratings.sample(frac=0.02)

print(small_data.info())

On utilise  la bibliothèque **scikit-learn** pour diviser l'ensemble de données en test et train.  La bibliothèque **Cross_validation.train_test_split** mélange et divise les données en deux ensembles de données en fonction du pourcentage d'exemples de test, qui dans ce cas est de 0,2.

In [ ]:

from sklearn.model_selection import train_test_split
train_data, test_data = train_test_split(small_data, test_size=0.2, random_state=42)

Nous devons créer une matrice utilisateur-item. Comme nous avons divisé les données en test et train, on doit créer deux matrices. La matrice de train contient 80 % des évaluations (ratings) et la matrice de test contient 20 % des évaluations.

In [ ]:
# Créer deux matrices utilisateur-item, l'une pour le training et l'autre pour le test.

# Convertir un DataFrame en tableau NumPy
train_data_matrix = train_data[['userId', 'movieId', 'rating']].to_numpy()
test_data_matrix = test_data[['userId', 'movieId', 'rating']].to_numpy()


print(train_data_matrix.shape)
print(test_data_matrix.shape)

La fonction **pairwise_distances** de sklearn permet de  calculer le [coefficient de corrélation de Pearson]. Cette méthode offre un moyen sûr de prendre une matrice de distance en entrée

In [ ]:
from sklearn.metrics.pairwise import pairwise_distances

# Matrice de similarité des users
user_correlation = 1 - pairwise_distances(train_data, metric='correlation')
user_correlation[np.isnan(user_correlation)] = 0
print(user_correlation[:4, :4])

In [ ]:
# Matrice de similarité des items
item_correlation = 1 - pairwise_distances(train_data_matrix.T, metric='correlation')
item_correlation[np.isnan(item_correlation)] = 0
print(item_correlation[:4, :4])

Avec la matrice de similarité on peut maintenant prédire les évaluations qui n'ont pas été incluses dans les données. À l'aide de ces prédictions, on peut ensuite les comparer aux données de test pour tenter de valider la qualité de notre modèle de recommandation.

Dans le cas du user-user CF, on considère la similarité entre deux utilisateurs (A et B, par exemple) comme des poids qui sont multipliés par les évaluations d'un utilisateur B similaire (corrigées en fonction de l'évaluation moyenne de cet utilisateur). On doit également normaliser le tout pour que les notes restent comprises entre 1 et 5 et, dans une dernière étape, additionner les notes moyennes de l'utilisateur que nous essayons de prédire. L'idée est que certains utilisateurs peuvent avoir tendance à donner des notes élevées ou faibles à tous les films. La différence relative entre les notes attribuées par ces utilisateurs est plus importante que les valeurs absolues.


In [ ]:
# Fonction pour prédir des notes
def predict(ratings, similarity, type='user'):
    if type == 'user':
        mean_user_rating = ratings.mean(axis=1)
        # Use np.newaxis so that mean_user_rating has same format as ratings
        ratings_diff = (ratings - mean_user_rating[:, np.newaxis])
        pred = mean_user_rating[:, np.newaxis] + similarity.dot(ratings_diff) / np.array([np.abs(similarity).sum(axis=1)]).T
    elif type == 'item':
        pred = ratings.dot(similarity) / np.array([np.abs(similarity).sum(axis=1)])
    return pred

### Evaluation
Il existe de nombreuses mesures d'évaluation, mais l'une des mesures les plus populaires utilisées pour évaluer la précision des évaluations prédites est l'**erreur quadratique moyenne de la racine (RMSE)**. Nous allons utiliser la fonction **mean_square_error (MSE)** de sklearn, où la RMSE est simplement la racine carrée de la MSE.

$$\mathit{RMSE} =\sqrt{\frac{1}{N} \sum (x_i -\hat{x_i})^2}$$.

On utilisera la fonction **moyenne des erreurs au carré** de scikit-learn comme mesure de validation. En comparant le filtrage collaboratif basé sur l'utilisateur et le filtrage collaboratif basé sur l'item.


In [ ]:
from sklearn.metrics import mean_squared_error
from math import sqrt

# Fonction de calcul de RMSE
def rmse(pred, actual):
    # Ignore nonzero terms.
    pred = pred[actual.nonzero()].flatten()
    actual = actual[actual.nonzero()].flatten()
    return sqrt(mean_squared_error(pred, actual))

In [ ]:
# Prédire les évaluations sur les données d'apprentissage avec les deux scores de similarité
user_prediction = predict(train_data_matrix, user_correlation, type='user')
item_prediction = predict(train_data_matrix, item_correlation, type='item')

# RMSE sur les données de test
print('User-based CF RMSE: ' + str(rmse(user_prediction, test_data_matrix)))
print('Item-based CF RMSE: ' + str(rmse(item_prediction, test_data_matrix)))

In [ ]:
# RMSE sur le train
print('User-based CF RMSE: ' + str(rmse(user_prediction, train_data_matrix)))
print('Item-based CF RMSE: ' + str(rmse(item_prediction, train_data_matrix)))

La RMSE dest une mesure qui permet d'évaluer dans quelle mesure le signal et le bruit sont expliqués par le modèle. Nous obtenons un RMSE  assez élevé lié à un sur-apprentissage.

Dans l'ensemble, le filtrage collaboratif basé sur la mémoire est facile à mettre en œuvre et produit une qualité de prédiction raisonnable. Cependant, cette approche présente certains inconvénients :

* Elle n'aborde pas le problème bien connu du démarrage à froid, c'est-à-dire lorsqu'un nouvel utilisateur ou un nouvel élément entre dans le système. 
* Elle ne peut pas traiter les données éparses, ce qui signifie qu'il est difficile de trouver des utilisateurs qui ont évalué les mêmes articles.
* Il souffre de l'arrivée de nouveaux utilisateurs ou d'éléments qui n'ont pas été évalués.
* Il a tendance à recommander des articles populaires.